In [7]:
import sys
sys.path.append("../src")

import importlib
import pandas as pd

import metrics
import backtesting
import bootstrap

importlib.reload(metrics)
importlib.reload(backtesting)
importlib.reload(bootstrap)

WINDOW = 3 * 252
STEP = 21
TRANSACTION_COST = 0.001
VOL_TARGET = 0.10
MAX_LEVERAGE = 2.0
FUNDING_SPREAD = 0.01

prices = pd.read_csv("../data/etf_prices_raw.csv", index_col="Date", parse_dates=True)
returns = metrics.compute_returns(prices)

rf_rate = pd.read_csv("../data/risk_free_rate.csv", index_col="Date", parse_dates=True)["risk_free_rate"]
rf_rate = pd.to_numeric(rf_rate, errors="coerce").ffill()

annual_rf = rf_rate / 100
risk_free_returns = (1 + annual_rf) ** (1 / 252) - 1
risk_free_returns = risk_free_returns.reindex(returns.index).ffill()

groups = {
    "equities": ["SPY", "QQQ", "EWU", "EZU", "EEM"],
    "bonds": ["IEF", "SHY"],
    "alternatives": ["GLD", "VNQ"]
}

strategies = {
    "Equal Weight": (backtesting.equal_strategy, None),
    "Inverse Vol": (backtesting.risk_strategy, None),
    "Max Geo": (backtesting.max_geo_strategy, None),
    "Max Sharpe": (backtesting.max_sharpe_strategy, None),
    "Carver": (backtesting.carver_handcrafting_strategy, groups),
    "Hierarchical Inv Vol": (backtesting.inv_handcrafting_strategy, groups),
    "SPY": (backtesting.spy_strategy, None),
    "60/40": (backtesting.sixty_forty_strategy, None),
    "Tech": (backtesting.tech_strategy, None)
}

raw_portfolio_returns = {}

for name, (strategy, strategy_groups) in strategies.items():
    port_ret, _, _, _, _ = backtesting.backtest(
        returns,
        window=WINDOW,
        step=STEP,
        allocation_func=strategy,
        groups=strategy_groups,
        vol_target=None,
        risk_free_returns=risk_free_returns,
        transaction_cost=TRANSACTION_COST,
        funding_spread=FUNDING_SPREAD
    )

    raw_portfolio_returns[name] = port_ret

targeted_portfolio_returns = {}

for name, (strategy, strategy_groups) in strategies.items():
    port_ret, _, _, _, _ = backtesting.backtest(
        returns,
        window=WINDOW,
        step=STEP,
        allocation_func=strategy,
        groups=strategy_groups,
        vol_target=VOL_TARGET,
        max_leverage=MAX_LEVERAGE,
        risk_free_returns=risk_free_returns,
        transaction_cost=TRANSACTION_COST,
        funding_spread=FUNDING_SPREAD
    )

    targeted_portfolio_returns[name] = port_ret

raw_returns_ts = pd.DataFrame(raw_portfolio_returns)
targeted_returns_ts = pd.DataFrame(targeted_portfolio_returns)

raw_excess = bootstrap.make_excess_returns(raw_returns_ts, risk_free_returns)
targeted_excess = bootstrap.make_excess_returns(targeted_returns_ts, risk_free_returns)

In [8]:
from itertools import combinations

BLOCK_SIZE = 21
N_BOOTSTRAP = 10_000
SEED = 42

In [9]:
results_raw = bootstrap.bootstrap_sharpe(
    raw_excess,
    block_size=BLOCK_SIZE,
    n_bootstrap=N_BOOTSTRAP,
    seed=SEED
)

confidence_intervals_raw = pd.DataFrame({
    "CI Lower": results_raw.quantile(0.025),
    "CI Upper": results_raw.quantile(0.975)
})

display(confidence_intervals_raw)

,CI Lower,CI Upper
Equal Weight,0.067657,0.903137
Inverse Vol,0.059799,0.960846
Max Geo,0.239936,1.101912
Max Sharpe,0.187373,1.009216
Carver,0.073721,0.984054
Hierarchical Inv Vol,0.074823,0.986715
SPY,0.190151,1.022573
60/40,0.230326,1.114479
Tech,0.326657,1.185076


In [10]:
comparisons_raw = []

for strat1, strat2 in combinations(results_raw.columns, 2):
    diff = results_raw[strat1] - results_raw[strat2]

    comparisons_raw.append({
        "Strategy 1": strat1,
        "Strategy 2": strat2,
        "Mean Diff": diff.mean(),
        "CI Lower": diff.quantile(0.025),
        "CI Upper": diff.quantile(0.975)
    })

comparison_table_raw = pd.DataFrame(comparisons_raw)

display(comparison_table_raw)

,Strategy 1,Strategy 2,Mean Diff,CI Lower,CI Upper
0,Equal Weight,Inverse Vol,-0.027100,-0.174148,0.125097
1,Equal Weight,Max Geo,-0.184926,-0.434345,0.070681
2,Equal Weight,Max Sharpe,-0.113706,-0.550651,0.306453
3,Equal Weight,Carver,-0.052735,-0.338528,0.239279
4,Equal Weight,Hierarchical Inv Vol,-0.053258,-0.326490,0.226112
5,Equal Weight,SPY,-0.116175,-0.287739,0.054720
6,Equal Weight,60/40,-0.184282,-0.348861,-0.021181
7,Equal Weight,Tech,-0.268242,-0.507112,-0.026019
8,Inverse Vol,Max Geo,-0.157827,-0.439907,0.122068
9,Inverse Vol,Max Sharpe,-0.086606,-0.495441,0.294328


In [11]:
results_targeted = bootstrap.bootstrap_sharpe(
    targeted_excess,
    block_size=BLOCK_SIZE,
    n_bootstrap=N_BOOTSTRAP,
    seed=SEED
)

confidence_intervals_targeted = pd.DataFrame({
    "CI Lower": results_targeted.quantile(0.025),
    "CI Upper": results_targeted.quantile(0.975)
})

display(confidence_intervals_targeted)

,CI Lower,CI Upper
Equal Weight,-0.020074,0.885279
Inverse Vol,-0.092620,0.809181
Max Geo,0.228125,1.108843
Max Sharpe,0.229569,1.072713
Carver,-0.072277,0.833730
Hierarchical Inv Vol,-0.065859,0.842005
SPY,0.113948,0.992810
60/40,0.156755,1.078406
Tech,0.281855,1.162269


In [12]:
comparisons_targeted = []

for strat1, strat2 in combinations(results_targeted.columns, 2):
    diff = results_targeted[strat1] - results_targeted[strat2]

    comparisons_targeted.append({
        "Strategy 1": strat1,
        "Strategy 2": strat2,
        "Mean Diff": diff.mean(),
        "CI Lower": diff.quantile(0.025),
        "CI Upper": diff.quantile(0.975)
    })

comparison_table_targeted = pd.DataFrame(comparisons_targeted)

display(comparison_table_targeted)

,Strategy 1,Strategy 2,Mean Diff,CI Lower,CI Upper
0,Equal Weight,Inverse Vol,0.067548,-0.116845,0.252786
1,Equal Weight,Max Geo,-0.241044,-0.446506,-0.020972
2,Equal Weight,Max Sharpe,-0.228326,-0.605228,0.147286
3,Equal Weight,Carver,0.038777,-0.278760,0.364313
4,Equal Weight,Hierarchical Inv Vol,0.030670,-0.273279,0.338964
5,Equal Weight,SPY,-0.116764,-0.304504,0.069809
6,Equal Weight,60/40,-0.180561,-0.349882,-0.009915
7,Equal Weight,Tech,-0.290898,-0.545130,-0.033141
8,Inverse Vol,Max Geo,-0.308593,-0.592590,-0.026340
9,Inverse Vol,Max Sharpe,-0.295874,-0.658874,0.061037
